In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
# 1. Lectura de datasets
train = pd.read_csv('./files/gold_recovery_train.csv')
test = pd.read_csv('./files/gold_recovery_test.csv')
full = pd.read_csv('./files/gold_recovery_full.csv')

train.info() # Mismo compartamiento para todos los datasets

# Cambio de tipo de fecha y establecimiento como índice para futuras iterpolaciones
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
full['date'] = pd.to_datetime(full['date'])

train = train.set_index('date')
test = test.set_index('date')
full = full.set_index('date')

In [19]:
# 2. Verificación de recuperación
def recovery(C, F, T):
    denom = F * (C - T)
    # Evitamos división por cero
    return np.where(denom == 0, np.nan, (C * (F - T)) / denom)

# Rougher recovery
rougher_calc = recovery(train['rougher.output.concentrate_au'],train['rougher.input.feed_au'],train['rougher.output.tail_au'])
eam_rougher = np.nanmean(np.abs(train['rougher.output.recovery'] - rougher_calc))
print("EAM Rougher:", eam_rougher)

# Final recovery
final_calc = recovery(train['final.output.concentrate_au'],train['rougher.output.concentrate_au'],train['final.output.tail_au'])
eam_final = np.nanmean(np.abs(train['final.output.recovery'] - final_calc))
print("EAM Final:", eam_final)

EAM Rougher: 81.57025919633583
EAM Final: 67.93959756401821


In [20]:
# 4. Columnas faltantes en test
missing_cols = set(train.columns) - set(test.columns)
print("Columnas faltantes en test:", missing_cols)

Columnas faltantes en test: {'primary_cleaner.output.tail_au', 'rougher.output.concentrate_au', 'rougher.output.concentrate_ag', 'rougher.output.tail_au', 'rougher.output.concentrate_pb', 'rougher.output.tail_sol', 'rougher.output.tail_ag', 'final.output.tail_ag', 'rougher.calculation.sulfate_to_au_concentrate', 'secondary_cleaner.output.tail_au', 'primary_cleaner.output.tail_sol', 'primary_cleaner.output.concentrate_sol', 'primary_cleaner.output.concentrate_au', 'rougher.output.concentrate_sol', 'final.output.tail_sol', 'primary_cleaner.output.concentrate_ag', 'final.output.recovery', 'final.output.concentrate_ag', 'final.output.concentrate_sol', 'primary_cleaner.output.tail_pb', 'primary_cleaner.output.tail_ag', 'secondary_cleaner.output.tail_sol', 'rougher.output.tail_pb', 'final.output.tail_pb', 'secondary_cleaner.output.tail_ag', 'final.output.concentrate_pb', 'rougher.calculation.floatbank11_sulfate_to_au_feed', 'secondary_cleaner.output.tail_pb', 'rougher.calculation.au_pb_ratio

In [26]:
# 5. Preprocesamiento de datos
# Eliminamos columnas objetivo del train para features
features_train = train.drop(['rougher.output.recovery','final.output.recovery'], axis=1)

# Relleno de nulos con interpolación temporal
features_train = train.drop(['rougher.output.recovery','final.output.recovery'], axis=1)
features_train = features_train.interpolate(method='time', limit_direction='forward')
features_test = test.interpolate(method='time', limit_direction='forward')

# Seleccionamos las columnas comunes entre train y test
common_cols = list(set(features_train.columns) & set(features_test.columns))

# Ordenamos para que coincidan exactamente
common_cols.sort()

# Creamos los datasets alineados
features_train_common = features_train[common_cols]
features_test_common = features_test[common_cols]

# Escalado
scaler = StandardScaler()
X_train = scaler.fit_transform(features_train_common)
X_test = scaler.transform(features_test_common)

y_train_rougher = train['rougher.output.recovery']
y_train_final = train['final.output.recovery']

In [27]:
# 6. Métrica sMAPE
def smape(y_true, y_pred):
    return np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred))) * 100

def smape_final(y_true_rougher, y_pred_rougher, y_true_final, y_pred_final):
    return 0.25 * smape(y_true_rougher, y_pred_rougher) + 0.75 * smape(y_true_final, y_pred_final)

smape_scorer = make_scorer(smape, greater_is_better=False)

In [32]:
# 7. Entrenamiento de modelos
# Eliminamos filas con NaN en las variables objetivo
train_clean = train.dropna(subset=['rougher.output.recovery','final.output.recovery'])

# Features y targets alineados
features_train_clean = train_clean.drop(['rougher.output.recovery','final.output.recovery'], axis=1)
features_train_clean = features_train_clean[common_cols]  # columnas comunes con test
features_test_common = features_test[common_cols]

# Imputación de nulos con la media
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(features_train_clean)
X_test = imputer.transform(features_test_common)

y_train_rougher = train_clean['rougher.output.recovery']
y_train_final = train_clean['final.output.recovery']

# Modelos a evaluar
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# Validación cruzada con sMAPE
for name, model in models.items():
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores_rougher = cross_val_score(model, X_train, y_train_rougher, cv=cv, scoring=smape_scorer)
    scores_final = cross_val_score(model, X_train, y_train_final, cv=cv, scoring=smape_scorer)
    print(f"{name} - sMAPE Rougher: {-scores_rougher.mean():.2f}, Final: {-scores_final.mean():.2f}")

LinearRegression - sMAPE Rougher: 10.16, Final: 9.09
RandomForest - sMAPE Rougher: 7.82, Final: 6.72


In [33]:
# 8. Entrenamiento final y predicción
best_model = RandomForestRegressor(n_estimators=200, random_state=42)
best_model.fit(X_train, y_train_final)
y_pred_final = best_model.predict(X_test)
print("Predicciones Final Recovery (primeros 10):", y_pred_final[:10])

Predicciones Final Recovery (primeros 10): [68.43897573 69.1670694  69.08089233 69.72101327 69.30444558 69.67711329
 63.63571812 64.02857535 65.170033   62.51259761]
